<a href="https://colab.research.google.com/github/talharaziq/amr-structural-epistasis_/blob/main/AMR_Structural_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define base directory structure in Drive
base_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project'
subdirectories = ['raw', 'processed', 'analysis', 'figures', 'manuscript']

# Create subdirectories
for folder in subdirectories:
    folder_path = os.path.join(base_dir, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Directory verified/created: {folder_path}")

print("\nStep 1 Complete: Google Drive mounted and directory structure established.")

Mounted at /content/drive
Directory verified/created: /content/drive/MyDrive/AMR_Structural_Mutation_Project/raw
Directory verified/created: /content/drive/MyDrive/AMR_Structural_Mutation_Project/processed
Directory verified/created: /content/drive/MyDrive/AMR_Structural_Mutation_Project/analysis
Directory verified/created: /content/drive/MyDrive/AMR_Structural_Mutation_Project/figures
Directory verified/created: /content/drive/MyDrive/AMR_Structural_Mutation_Project/manuscript

Step 1 Complete: Google Drive mounted and directory structure established.


In [3]:
import os
import urllib.request
import tarfile

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
tar_path = os.path.join(raw_dir, 'card-data.tar.bz2')

# 1. Download the latest CARD dataset
url = 'https://card.mcmaster.ca/latest/data'
print("Downloading CARD dataset from card.mcmaster.ca...")
urllib.request.urlretrieve(url, tar_path)
print(f"Downloaded: {tar_path} ({os.path.getsize(tar_path)} bytes)")

# 2. Extract contents into raw folder without modifying raw tarball
print("Extracting files...")
with tarfile.open(tar_path, "r:bz2") as tar:
    tar.extractall(path=raw_dir)

# 3. List extracted files for inspection
print("\nExtracted raw CARD dataset files:")
for filename in sorted(os.listdir(raw_dir)):
    filepath = os.path.join(raw_dir, filename)
    if os.path.isfile(filepath):
        print(f" - {filename} ({os.path.getsize(filepath)} bytes)")

Downloaded: /content/drive/MyDrive/AMR_Structural_Mutation_Project/raw/card-data.tar.bz2 (4366637 bytes)
Extracting files...


/tmp/ipykernel_958/468984039.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=raw_dir)



Extracted raw CARD dataset files:
 - CARD-Download-README.txt (4024 bytes)
 - PMID.tsv (220995 bytes)
 - aro_categories.tsv (37884 bytes)
 - aro_categories_index.tsv (735859 bytes)
 - aro_index.tsv (1127760 bytes)
 - card-data.tar.bz2 (4366637 bytes)
 - card.json (38086878 bytes)
 - nucleotide_fasta_protein_homolog_model.fasta (6285453 bytes)
 - nucleotide_fasta_protein_knockout_model.fasta (16904 bytes)
 - nucleotide_fasta_protein_overexpression_model.fasta (11562 bytes)
 - nucleotide_fasta_protein_variant_model.fasta (429485 bytes)
 - nucleotide_fasta_rRNA_gene_variant_model.fasta (186540 bytes)
 - protein_fasta_protein_homolog_model.fasta (2332067 bytes)
 - protein_fasta_protein_knockout_model.fasta (6261 bytes)
 - protein_fasta_protein_overexpression_model.fasta (4516 bytes)
 - protein_fasta_protein_variant_model.fasta (154021 bytes)
 - shortname_antibiotics.tsv (1352 bytes)
 - shortname_pathogens.tsv (2726 bytes)
 - snps.txt (648676 bytes)


In [4]:
import json
import os

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
json_path = os.path.join(raw_dir, 'card.json')

# Load card.json
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

print(f"Total keys in card.json: {len(card_data)}")

# Classify entry types and identify variant models
model_types = set()
variant_entries = []

for key, entry in card_data.items():
    if isinstance(entry, dict):
        m_type = entry.get('model_type')
        if m_type:
            model_types.add(m_type)
        if m_type and 'variant' in m_type.lower():
            variant_entries.append(entry)

print("Discovered model types in CARD:", model_types)
print(f"Number of variant model entries found: {len(variant_entries)}")

# Print structural details of a sample variant model
if variant_entries:
    sample = variant_entries[0]
    print("\nSample Variant Model Inspection:")
    print("ARO Accession:", sample.get('ARO_accession'))
    print("ARO Name:", sample.get('ARO_name'))
    print("Top-level keys:", list(sample.keys()))
    if 'model_param' in sample:
        print("model_param keys:", list(sample['model_param'].keys()))

Total keys in card.json: 6454
Discovered model types in CARD: {'protein overexpression model', 'protein homolog model', 'protein variant model', 'gene cluster meta-model', 'efflux pump system meta-model', 'rRNA gene variant model', 'protein knockout model'}
Number of variant model entries found: 325

Sample Variant Model Inspection:
ARO Accession: 3003582
ARO Name: basR
Top-level keys: ['model_id', 'model_name', 'model_type', 'model_type_id', 'model_description', 'model_param', 'model_sequences', 'ARO_accession', 'ARO_id', 'ARO_name', 'CARD_short_name', 'ARO_description', 'ARO_category']
model_param keys: ['snp', 'blastp_bit_score']


In [5]:
import json
import os
import pandas as pd
from itertools import combinations

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
json_path = os.path.join(raw_dir, 'card.json')

with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

candidate_pairs = []
pair_counter = 1

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    m_type = entry.get('model_type', '')
    if 'variant' in m_type.lower():
        model_param = entry.get('model_param', {})
        snp_param = model_param.get('snp', {})

        # Extract SNP list from model_param
        snps = []
        if isinstance(snp_param, dict):
            param_val = snp_param.get('param_value', {})
            if isinstance(param_val, dict):
                # Common CARD structure for variant models
                for key, val in param_val.items():
                    if isinstance(val, list):
                        snps.extend(val)
                    elif isinstance(val, str):
                        snps.append(val)
            elif isinstance(param_val, list):
                snps = param_val

        # Clean snps list and keep unique mutation entries per model
        snps = sorted(list(set([str(s).strip() for s in snps if s])))

        # Extract pairs if 2 or more mutations exist in the model
        if len(snps) >= 2:
            gene_name = entry.get('ARO_name', 'Unknown')
            card_model = entry.get('model_name', 'Unknown')

            # Extract organism and drug information if available in categories
            organism = "Unspecified"
            drug = "Unspecified"
            categories = entry.get('ARO_category', {})
            if isinstance(categories, dict):
                for cat_id, cat_info in categories.items():
                    cat_name = cat_info.get('category_aro_name', '')
                    cat_cvterm = cat_info.get('category_aro_cvterm_id', '')
                    if 'resistance mechanism' in cat_name.lower():
                        continue
                    elif 'drug' in cat_name.lower() or 'antibiotic' in cat_name.lower():
                        drug = cat_name
                    else:
                        organism = cat_name

            # Generate pair-level records for co-occurring/co-required mutations
            for m1, m2 in combinations(snps, 2):
                candidate_pairs.append({
                    'Pair_id': f'PAIR_{pair_counter:04d}',
                    'Protein/gene': gene_name,
                    'Mutation_1': m1,
                    'Mutation_2': m2,
                    'Organism': organism,
                    'Drug': drug,
                    'CARD_model': card_model
                })
                pair_counter += 1

df_pairs = pd.DataFrame(candidate_pairs)
out_path = os.path.join(processed_dir, 'candidate_pairs.csv')
df_pairs.to_csv(out_path, index=False)

print(f"Total candidate mutation pairs extracted: {len(df_pairs)}")
if not df_pairs.empty:
    print("\nSample candidate pairs:")
    print(df_pairs.head(10).to_string())

Total candidate mutation pairs extracted: 121325

Sample candidate pairs:
     Pair_id                                                               Protein/gene Mutation_1 Mutation_2                                             Organism                          Drug                                                                 CARD_model
0  PAIR_0001                                                                       basR      L157Q       L71R                  pmr phosphoethanolamine transferase             antibiotic efflux                                                                       basR
1  PAIR_0002        Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin      N609D      S575T                               penicillin beta-lactam  antibiotic target alteration        Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
2  PAIR_0003  Escherichia coli folP with mutation conferring resistance to sulfonamides       P64A       P64H  sulfonamide

In [6]:
import os
import re
import pandas as pd

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
in_path = os.path.join(processed_dir, 'candidate_pairs.csv')
df = pd.read_csv(in_path)

candidate_count = len(df)

# Standard single-letter amino acid mutation pattern (WT, POS, MUT)
aa_pattern = re.compile(r'^([ACDEFGHIKLMNPQRSTVWY])(\d+)([ACDEFGHIKLMNPQRSTVWY])$')

usable_pairs = []
excluded_records = []

for idx, row in df.iterrows():
    m1 = str(row['Mutation_1']).strip()
    m2 = str(row['Mutation_2']).strip()
    pair_id = row['Pair_id']

    match1 = aa_pattern.match(m1)
    match2 = aa_pattern.match(m2)

    if not match1 or not match2:
        reason = "Non-standard or non-amino acid mutation notation"
        excluded_records.append({'Pair_id': pair_id, 'Reason_excluded': reason, 'Mutation_1': m1, 'Mutation_2': m2})
        continue

    wt1, pos1, mut1 = match1.groups()
    wt2, pos2, mut2 = match2.groups()
    pos1, pos2 = int(pos1), int(pos2)

    if pos1 == pos2:
        reason = "Same residue position (alternative single-point alleles)"
        excluded_records.append({'Pair_id': pair_id, 'Reason_excluded': reason, 'Mutation_1': m1, 'Mutation_2': m2})
        continue

    row_dict = row.to_dict()
    row_dict['WT_1'] = wt1
    row_dict['Pos_1'] = pos1
    row_dict['Mut_1'] = mut1
    row_dict['WT_2'] = wt2
    row_dict['Pos_2'] = pos2
    row_dict['Mut_2'] = mut2
    usable_pairs.append(row_dict)

df_usable = pd.DataFrame(usable_pairs)
df_excluded = pd.DataFrame(excluded_records)

df_usable.to_csv(os.path.join(processed_dir, 'usable_pairs.csv'), index=False)
df_excluded.to_csv(os.path.join(processed_dir, 'excluded_pairs_step4.csv'), index=False)

print(f"Total candidate pairs evaluated: {candidate_count}")
print(f"Total usable same-protein pairs retained: {len(df_usable)}")
print(f"Total pairs excluded: {len(df_excluded)}")
print("\nBreakdown of exclusion reasons:")
print(df_excluded['Reason_excluded'].value_counts().to_string())
if not df_usable.empty:
    print("\nSample retained usable pair:")
    print(df_usable.head(5)[['Pair_id', 'Protein/gene', 'WT_1', 'Pos_1', 'Mut_1', 'WT_2', 'Pos_2', 'Mut_2']].to_string())

Total candidate pairs evaluated: 121325
Total usable same-protein pairs retained: 114068
Total pairs excluded: 7257

Breakdown of exclusion reasons:
Reason_excluded
Non-standard or non-amino acid mutation notation            5934
Same residue position (alternative single-point alleles)    1323

Sample retained usable pair:
     Pair_id                                                            Protein/gene WT_1  Pos_1 Mut_1 WT_2  Pos_2 Mut_2
0  PAIR_0001                                                                    basR    L    157     Q    L     71     R
1  PAIR_0002     Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin    N    609     D    S    575     T
2  PAIR_0013  Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473     T    A    477     T
3  PAIR_0014  Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473     T    H    481     N
4  PAIR_0015  Staphylococcus aureus rpoB mutants conferring resistance

In [7]:
import json
import os
import pandas as pd

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'

# 1. Load card.json to map CARD models to their reference protein sequences
json_path = os.path.join(raw_dir, 'card.json')
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

model_seq_map = {}
for model_key, entry in card_data.items():
    if isinstance(entry, dict) and 'model_sequences' in entry:
        model_name = entry.get('model_name')
        aro_name = entry.get('ARO_name')
        seq_data = entry.get('model_sequences', {})
        protein_seqs = []
        if isinstance(seq_data, dict):
            for seq_key, seq_val in seq_data.items():
                if isinstance(seq_val, dict):
                    p_seq = seq_val.get('protein_sequence', {}).get('sequence')
                    if p_seq:
                        protein_seqs.append(p_seq)
        if protein_seqs:
            if model_name:
                model_seq_map[model_name] = protein_seqs[0]
            if aro_name:
                model_seq_map[aro_name] = protein_seqs[0]

# 2. Load usable pairs from Step 4
usable_path = os.path.join(processed_dir, 'usable_pairs.csv')
df_usable = pd.read_csv(usable_path)

verified_pairs = []
excluded_step5 = []

# 3. Verify wild-type residue match and sequence boundary
for idx, row in df_usable.iterrows():
    pair_id = row['Pair_id']
    model_name = row['CARD_model']
    gene_name = row['Protein/gene']

    wt1, pos1 = row['WT_1'], int(row['Pos_1'])
    wt2, pos2 = row['WT_2'], int(row['Pos_2'])

    seq = model_seq_map.get(model_name) or model_seq_map.get(gene_name)

    if not seq:
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': 'Reference sequence not found in CARD model_sequences',
            'CARD_model': model_name
        })
        continue

    seq_len = len(seq)

    if pos1 < 1 or pos1 > seq_len or pos2 < 1 or pos2 > seq_len:
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': f'Mutation position out of bounds (Seq len: {seq_len})',
            'CARD_model': model_name
        })
        continue

    ref_wt1 = seq[pos1 - 1]
    ref_wt2 = seq[pos2 - 1]

    if ref_wt1 != wt1 or ref_wt2 != wt2:
        mismatch_details = f"Pos1 {wt1}->{ref_wt1}" if ref_wt1 != wt1 else ""
        mismatch_details += f" Pos2 {wt2}->{ref_wt2}" if ref_wt2 != wt2 else ""
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': f'WT amino acid mismatch ({mismatch_details.strip()})',
            'CARD_model': model_name
        })
        continue

    row_dict = row.to_dict()
    row_dict['Ref_Sequence'] = seq
    row_dict['Seq_Length'] = seq_len
    verified_pairs.append(row_dict)

df_verified = pd.DataFrame(verified_pairs)
df_excluded5 = pd.DataFrame(excluded_step5)

df_verified.to_csv(os.path.join(processed_dir, 'verified_pairs.csv'), index=False)
df_excluded5.to_csv(os.path.join(processed_dir, 'excluded_pairs_step5.csv'), index=False)

print(f"Usable pairs evaluated: {len(df_usable)}")
print(f"Verified mutation pairs retained: {len(df_verified)}")
print(f"Excluded in Step 5: {len(df_excluded5)}")
if not df_excluded5.empty:
    print("\nExclusion breakdown in Step 5:")
    print(df_excluded5['Reason_excluded'].value_counts().to_string())
if not df_verified.empty:
    print("\nSample verified pair:")
    print(df_verified.head(3)[['Pair_id', 'Protein/gene', 'WT_1', 'Pos_1', 'WT_2', 'Pos_2', 'Seq_Length']].to_string())

Usable pairs evaluated: 114068
Verified mutation pairs retained: 0
Excluded in Step 5: 114068

Exclusion breakdown in Step 5:
Reason_excluded
Reference sequence not found in CARD model_sequences    114068


In [8]:
import json
import os

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
json_path = os.path.join(raw_dir, 'card.json')

with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

# Inspect model_sequences for 3 variant model entries
print("--- Inspecting CARD JSON sequence fields ---")
sample_count = 0
for model_key, entry in card_data.items():
    if isinstance(entry, dict) and 'variant' in entry.get('model_type', '').lower():
        sample_count += 1
        print(f"\nVariant Model #{sample_count}:")
        print(f"  Key: {model_key}")
        print(f"  ARO_accession: {entry.get('ARO_accession')}")
        print(f"  ARO_name: {entry.get('ARO_name')}")
        print(f"  model_name: {entry.get('model_name')}")

        seqs = entry.get('model_sequences', {})
        print(f"  model_sequences keys type: {type(seqs)}")
        if isinstance(seqs, dict):
            print(f"  model_sequences keys count: {len(seqs)}")
            first_seq_key = next(iter(seqs.keys()), None)
            if first_seq_key:
                print(f"  Sample seq key: {first_seq_key}")
                print(f"  Sample seq value keys: {list(seqs[first_seq_key].keys()) if isinstance(seqs[first_seq_key], dict) else type(seqs[first_seq_key])}")

        if sample_count >= 3:
            break

--- Inspecting CARD JSON sequence fields ---

Variant Model #1:
  Key: 632
  ARO_accession: 3003582
  ARO_name: basR
  model_name: basR
  model_sequences keys type: <class 'dict'>
  model_sequences keys count: 1
  Sample seq key: sequence
  Sample seq value keys: ['5429']

Variant Model #2:
  Key: 1607
  ARO_accession: 3003041
  ARO_name: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  model_name: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  model_sequences keys type: <class 'dict'>
  model_sequences keys count: 1
  Sample seq key: sequence
  Sample seq value keys: ['692']

Variant Model #3:
  Key: 1966
  ARO_accession: 3003028
  ARO_name: lmrA
  model_name: lmrA
  model_sequences keys type: <class 'dict'>
  model_sequences keys count: 1
  Sample seq key: sequence
  Sample seq value keys: ['5186']


In [9]:
import json
import os
import pandas as pd

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'

# 1. Load card.json and extract protein sequence mappings
json_path = os.path.join(raw_dir, 'card.json')
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

model_seq_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    m_name = entry.get('model_name')
    aro_name = entry.get('ARO_name')
    model_id = entry.get('model_id')

    seqs_dict = entry.get('model_sequences', {})
    if isinstance(seqs_dict, dict) and 'sequence' in seqs_dict:
        seq_items = seqs_dict['sequence']
        if isinstance(seq_items, dict):
            for seq_id, seq_obj in seq_items.items():
                if isinstance(seq_obj, dict):
                    prot_obj = seq_obj.get('protein_sequence', {})
                    if isinstance(prot_obj, dict):
                        p_seq = prot_obj.get('sequence')
                        if p_seq:
                            if m_name:
                                model_seq_map[m_name] = p_seq
                            if aro_name:
                                model_seq_map[aro_name] = p_seq
                            if model_id:
                                model_seq_map[str(model_id)] = p_seq
                            break

print(f"Extracted reference protein sequences for {len(model_seq_map)} model identifiers.")

# 2. Load usable pairs from Step 4
usable_path = os.path.join(processed_dir, 'usable_pairs.csv')
df_usable = pd.read_csv(usable_path)

verified_pairs = []
excluded_step5 = []

# 3. Verify wild-type residue match and sequence boundary
for idx, row in df_usable.iterrows():
    pair_id = row['Pair_id']
    model_name = row['CARD_model']
    gene_name = row['Protein/gene']

    wt1, pos1 = str(row['WT_1']).strip(), int(row['Pos_1'])
    wt2, pos2 = str(row['WT_2']).strip(), int(row['Pos_2'])

    seq = model_seq_map.get(model_name) or model_seq_map.get(gene_name)

    if not seq:
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': 'Reference sequence not found in CARD JSON',
            'CARD_model': model_name
        })
        continue

    seq_len = len(seq)

    if pos1 < 1 or pos1 > seq_len or pos2 < 1 or pos2 > seq_len:
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': f'Mutation position out of bounds (Seq len: {seq_len})',
            'CARD_model': model_name
        })
        continue

    ref_wt1 = seq[pos1 - 1]
    ref_wt2 = seq[pos2 - 1]

    if ref_wt1 != wt1 or ref_wt2 != wt2:
        mismatch_details = f"Pos1 {wt1}->{ref_wt1}" if ref_wt1 != wt1 else ""
        mismatch_details += f" Pos2 {wt2}->{ref_wt2}" if ref_wt2 != wt2 else ""
        excluded_step5.append({
            'Pair_id': pair_id,
            'Reason_excluded': f'WT amino acid mismatch ({mismatch_details.strip()})',
            'CARD_model': model_name
        })
        continue

    row_dict = row.to_dict()
    row_dict['Ref_Sequence'] = seq
    row_dict['Seq_Length'] = seq_len
    verified_pairs.append(row_dict)

df_verified = pd.DataFrame(verified_pairs)
df_excluded5 = pd.DataFrame(excluded_step5)

df_verified.to_csv(os.path.join(processed_dir, 'verified_pairs.csv'), index=False)
df_excluded5.to_csv(os.path.join(processed_dir, 'excluded_pairs_step5.csv'), index=False)

print(f"Usable pairs evaluated: {len(df_usable)}")
print(f"Verified mutation pairs retained: {len(df_verified)}")
print(f"Excluded in Step 5: {len(df_excluded5)}")
if not df_excluded5.empty:
    print("\nExclusion breakdown in Step 5:")
    print(df_excluded5['Reason_excluded'].value_counts().to_string())
if not df_verified.empty:
    print("\nSample verified pair:")
    print(df_verified.head(3)[['Pair_id', 'Protein/gene', 'WT_1', 'Pos_1', 'WT_2', 'Pos_2', 'Seq_Length']].to_string())

Extracted reference protein sequences for 12658 model identifiers.
Usable pairs evaluated: 114068
Verified mutation pairs retained: 113398
Excluded in Step 5: 670

Exclusion breakdown in Step 5:
Reason_excluded
WT amino acid mismatch (Pos2 V->Y)              29
WT amino acid mismatch (Pos1 F->T)              26
WT amino acid mismatch (Pos2 T->F)              23
WT amino acid mismatch (Pos1 N->K)              22
WT amino acid mismatch (Pos1 F->K)              19
WT amino acid mismatch (Pos1 I->S)              17
WT amino acid mismatch (Pos2 T->A)              14
WT amino acid mismatch (Pos2 N->K)              14
WT amino acid mismatch (Pos1 A->S)              12
WT amino acid mismatch (Pos2 I->S)              12
WT amino acid mismatch (Pos2 V->L)              11
WT amino acid mismatch (Pos2 Y->D)              11
WT amino acid mismatch (Pos1 G->C)              11
WT amino acid mismatch (Pos1 C->A)              10
WT amino acid mismatch (Pos2 T->L)              10
WT amino acid mismatch (

In [10]:
import os
import json
import requests
import pandas as pd

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
struct_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw/structures'
os.makedirs(struct_dir, exist_ok=True)

# 1. Load verified pairs
verified_path = os.path.join(processed_dir, 'verified_pairs.csv')
df = pd.read_csv(verified_path)

# Extract unique models/genes
unique_models = df[['CARD_model', 'Protein/gene', 'Seq_Length']].drop_duplicates().to_dict('records')
print(f"Total verified mutation pairs: {len(df)}")
print(f"Total unique protein models to map structures for: {len(unique_models)}")

# 2. Extract UniProt accessions from card.json for structure retrieval
json_path = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw/card.json'
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

model_uniprot_map = {}
for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue
    m_name = entry.get('model_name')
    aro_name = entry.get('ARO_name')

    # Search for UniProt ID in model parameters or descriptions
    uniprot_id = None
    # Check ARO categories or model params
    seqs_dict = entry.get('model_sequences', {})
    if isinstance(seqs_dict, dict) and 'sequence' in seqs_dict:
        for seq_id, seq_obj in seqs_dict['sequence'].items():
            if isinstance(seq_obj, dict):
                acc = seq_obj.get('protein_sequence', {}).get('accession', '')
                if acc and ('_' not in acc) and (len(acc) in [6, 10]):
                    uniprot_id = acc
                    break
    if uniprot_id:
        if m_name: model_uniprot_map[m_name] = uniprot_id
        if aro_name: model_uniprot_map[aro_name] = uniprot_id

# Function to fetch structure (PDB or AlphaFold)
structure_registry = {}

for item in unique_models:
    model_name = item['CARD_model']
    gene_name = item['Protein/gene']
    seq_len = item['Seq_Length']

    uniprot_acc = model_uniprot_map.get(model_name) or model_uniprot_map.get(gene_name)

    struct_id = None
    struct_source = None
    pdb_file_path = None
    chain = 'A'

    # Try AlphaFold DB if UniProt ID exists
    if uniprot_acc:
        af_url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_acc}-F1-model_v4.pdb"
        resp = requests.get(af_url)
        if resp.status_code == 200:
            struct_id = f"AF-{uniprot_acc}-F1"
            struct_source = "AlphaFold"
            pdb_file_path = os.path.join(struct_dir, f"{struct_id}.pdb")
            with open(pdb_file_path, 'w') as f_out:
                f_out.write(resp.text)

    structure_registry[model_name] = {
        'UniProt': uniprot_acc,
        'Structure_id': struct_id,
        'Structure_source': struct_source,
        'PDB_path': pdb_file_path,
        'Chain': chain
    }

print("\nStructure lookup completed for unique protein models.")
mapped_count = sum(1 for v in structure_registry.values() if v['Structure_id'] is not None)
print(f"Models with retrieved structure: {mapped_count} / {len(unique_models)}")

Total verified mutation pairs: 113398
Total unique protein models to map structures for: 129

Structure lookup completed for unique protein models.
Models with retrieved structure: 0 / 129


In [11]:
import json
import os

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
json_path = os.path.join(raw_dir, 'card.json')

with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

# Inspect sequence accessions and metadata for 5 sample models
sample_count = 0
print("--- Inspecting sequence metadata in card.json ---")
for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    m_type = entry.get('model_type', '')
    if 'variant' in m_type.lower():
        sample_count += 1
        print(f"\nModel #{sample_count}: {entry.get('model_name')}")
        seqs_dict = entry.get('model_sequences', {})
        if isinstance(seqs_dict, dict) and 'sequence' in seqs_dict:
            for seq_id, seq_obj in list(seqs_dict['sequence'].items())[:2]:
                if isinstance(seq_obj, dict):
                    prot_obj = seq_obj.get('protein_sequence', {})
                    print("  seq_id:", seq_id)
                    print("  protein_sequence keys:", list(prot_obj.keys()))
                    print("  accession:", prot_obj.get('accession'))
                    print("  header/description:", prot_obj.get('description'))

        if sample_count >= 5:
            break

--- Inspecting sequence metadata in card.json ---

Model #1: basR
  seq_id: 5429
  protein_sequence keys: ['accession', 'sequence']
  accession: AAG08162.1
  header/description: None

Model #2: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  seq_id: 692
  protein_sequence keys: ['accession', 'sequence']
  accession: AFC91828.1
  header/description: None

Model #3: lmrA
  seq_id: 5186
  protein_sequence keys: ['accession', 'sequence']
  accession: CAA42550.1
  header/description: None

Model #4: Escherichia coli folP with mutation conferring resistance to sulfonamides
  seq_id: 5412
  protein_sequence keys: ['accession', 'sequence']
  accession: AAC76209.2
  header/description: None

Model #5: mtrR
  seq_id: 4254
  protein_sequence keys: ['accession', 'sequence']
  accession: ACF30254.1
  header/description: None


In [12]:
import os
import json
import time
import requests
import pandas as pd

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
struct_dir = os.path.join(raw_dir, 'structures')
os.makedirs(struct_dir, exist_ok=True)

# 1. Load verified pairs
verified_path = os.path.join(processed_dir, 'verified_pairs.csv')
df = pd.read_csv(verified_path)

# 2. Extract NCBI protein accessions from card.json
json_path = os.path.join(raw_dir, 'card.json')
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

model_ncbi_map = {}
for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue
    m_name = entry.get('model_name')
    aro_name = entry.get('ARO_name')
    model_id = entry.get('model_id')

    seqs_dict = entry.get('model_sequences', {})
    if isinstance(seqs_dict, dict) and 'sequence' in seqs_dict:
        for seq_id, seq_obj in seqs_dict['sequence'].items():
            if isinstance(seq_obj, dict):
                prot_obj = seq_obj.get('protein_sequence', {})
                acc = prot_obj.get('accession')
                if acc:
                    acc_clean = acc.split('.')[0]
                    if m_name: model_ncbi_map[m_name] = acc_clean
                    if aro_name: model_ncbi_map[aro_name] = acc_clean
                    if model_id: model_ncbi_map[str(model_id)] = acc_clean
                    break

# 3. Extract 129 unique models
unique_models = df[['CARD_model', 'Protein/gene', 'Seq_Length']].drop_duplicates()
print(f"Total unique protein models: {len(unique_models)}")

# 4. Map NCBI accessions to UniProt and download AlphaFold DB PDB structures
model_structure_info = {}

print("Mapping NCBI accessions to UniProt and downloading 3D structures...")
for idx, row in unique_models.iterrows():
    m_name = row['CARD_model']
    gene_name = row['Protein/gene']
    seq_len = row['Seq_Length']

    ncbi_acc = model_ncbi_map.get(m_name) or model_ncbi_map.get(gene_name)

    uniprot_acc = None
    struct_id = None
    struct_source = None
    pdb_path = None
    chain = 'A'

    if ncbi_acc:
        try:
            url = f"https://rest.uniprot.org/uniprotkb/search?query={ncbi_acc}&format=json&size=1"
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                results = data.get('results', [])
                if results:
                    uniprot_acc = results[0].get('primaryAccession')
        except Exception:
            pass

    if uniprot_acc:
        af_url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_acc}-F1-model_v4.pdb"
        try:
            af_res = requests.get(af_url, timeout=10)
            if af_res.status_code == 200:
                struct_id = f"AF-{uniprot_acc}-F1"
                struct_source = "AlphaFold"
                pdb_path = os.path.join(struct_dir, f"{struct_id}.pdb")
                with open(pdb_path, 'w') as f_out:
                    f_out.write(af_res.text)
        except Exception:
            pass

    model_structure_info[m_name] = {
        'NCBI_Acc': ncbi_acc,
        'UniProt_Acc': uniprot_acc,
        'Structure_id': struct_id,
        'Structure_source': struct_source,
        'PDB_path': pdb_path,
        'Chain': chain
    }

# Save structure mapping table
df_struct_map = pd.DataFrame.from_dict(model_structure_info, orient='index').reset_index()
df_struct_map.rename(columns={'index': 'CARD_model'}, inplace=True)
df_struct_map.to_csv(os.path.join(processed_dir, 'structure_mapping.csv'), index=False)

mapped_models = sum(1 for v in model_structure_info.values() if v['Structure_id'] is not None)
print(f"\nStructure lookup complete:")
print(f"Unique models successfully mapped to 3D structures: {mapped_models} / {len(unique_models)}")

Total unique protein models: 129
Mapping NCBI accessions to UniProt and downloading 3D structures...

Structure lookup complete:
Unique models successfully mapped to 3D structures: 0 / 129


In [13]:
import requests

test_accessions = ['AAG08162.1', 'AAG08162', 'AAC76209.2', 'AAC76209', 'AFC91828.1']

print("--- Testing UniProt API search syntax ---")
for acc in test_accessions:
    # 1. Broad search
    url1 = f"https://rest.uniprot.org/uniprotkb/search?query={acc}&format=json&size=1"
    r1 = requests.get(url1).json()
    res1 = r1.get('results', [])
    u1 = res1[0]['primaryAccession'] if res1 else "None"

    # 2. Database cross-reference search
    url2 = f"https://rest.uniprot.org/uniprotkb/search?query=xref:{acc}&format=json&size=1"
    r2 = requests.get(url2).json()
    res2 = r2.get('results', [])
    u2 = res2[0]['primaryAccession'] if res2 else "None"

    print(f"Accession: {acc:12s} | General Query: {u1:10s} | xref Query: {u2:10s}")

--- Testing UniProt API search syntax ---
Accession: AAG08162.1   | General Query: Q9HV32     | xref Query: Q9HV32    
Accession: AAG08162     | General Query: Q9HV32     | xref Query: Q9HV32    
Accession: AAC76209.2   | General Query: P0AC13     | xref Query: P0AC13    
Accession: AAC76209     | General Query: P0AC13     | xref Query: P0AC13    
Accession: AFC91828.1   | General Query: H8ZYN4     | xref Query: H8ZYN4    


In [14]:
import os
import json
import requests
import pandas as pd

raw_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/raw'
processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
struct_dir = os.path.join(raw_dir, 'structures')
os.makedirs(struct_dir, exist_ok=True)

# 1. Load verified pairs
verified_path = os.path.join(processed_dir, 'verified_pairs.csv')
df_verified = pd.read_csv(verified_path)

# 2. Extract NCBI accessions from card.json
json_path = os.path.join(raw_dir, 'card.json')
with open(json_path, 'r', encoding='utf-8') as f:
    card_data = json.load(f)

model_ncbi_map = {}
for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue
    m_name = entry.get('model_name')
    aro_name = entry.get('ARO_name')
    model_id = entry.get('model_id')

    seqs_dict = entry.get('model_sequences', {})
    if isinstance(seqs_dict, dict) and 'sequence' in seqs_dict:
        for seq_id, seq_obj in seqs_dict['sequence'].items():
            if isinstance(seq_obj, dict):
                prot_obj = seq_obj.get('protein_sequence', {})
                acc = prot_obj.get('accession')
                if acc:
                    if m_name: model_ncbi_map[m_name] = acc
                    if aro_name: model_ncbi_map[aro_name] = acc
                    if model_id: model_ncbi_map[str(model_id)] = acc
                    break

# 3. Process unique protein models
unique_models = df_verified[['CARD_model', 'Protein/gene', 'Seq_Length']].drop_duplicates().to_dict('records')
print(f"Total unique protein models: {len(unique_models)}")

model_struct_map = {}

for item in unique_models:
    m_name = item['CARD_model']
    gene_name = item['Protein/gene']
    seq_len = item['Seq_Length']
    ncbi_acc = model_ncbi_map.get(m_name) or model_ncbi_map.get(gene_name)

    uniprot_acc = None
    struct_id = None
    struct_source = None
    pdb_path = None
    chain = 'A'

    if ncbi_acc:
        # Search UniProt KB using xref query
        try:
            u_url = f"https://rest.uniprot.org/uniprotkb/search?query=xref:{ncbi_acc}&format=json&size=1"
            u_res = requests.get(u_url, timeout=10).json()
            results = u_res.get('results', [])
            if results:
                uniprot_acc = results[0].get('primaryAccession')
        except Exception:
            pass

    if uniprot_acc:
        # Query AlphaFold EBI API to get exact PDB download URL
        try:
            af_api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_acc}"
            af_res = requests.get(af_api_url, timeout=10)
            if af_res.status_code == 200:
                af_data = af_res.json()
                if isinstance(af_data, list) and len(af_data) > 0:
                    pdb_url = af_data[0].get('pdbUrl')
                    if pdb_url:
                        struct_id = f"AF-{uniprot_acc}"
                        struct_source = "AlphaFold"
                        pdb_path = os.path.join(struct_dir, f"{struct_id}.pdb")
                        p_file = requests.get(pdb_url, timeout=15)
                        if p_file.status_code == 200:
                            with open(pdb_path, 'w') as f_out:
                                f_out.write(p_file.text)
        except Exception:
            pass

    model_struct_map[m_name] = {
        'NCBI_Acc': ncbi_acc,
        'UniProt_Acc': uniprot_acc,
        'Structure_id': struct_id,
        'Structure_source': struct_source,
        'PDB_path': pdb_path,
        'Chain': chain
    }

mapped_count = sum(1 for v in model_struct_map.values() if v['Structure_id'] is not None and v['PDB_path'] and os.path.exists(v['PDB_path']))
print(f"\nStructure lookup completed: {mapped_count} / {len(unique_models)} models successfully retrieved 3D structure PDB files.")

# Save structure mapping table
df_struct = pd.DataFrame.from_dict(model_struct_map, orient='index').reset_index()
df_struct.rename(columns={'index': 'CARD_model'}, inplace=True)
df_struct.to_csv(os.path.join(processed_dir, 'structure_mapping.csv'), index=False)

Total unique protein models: 129

Structure lookup completed: 113 / 129 models successfully retrieved 3D structure PDB files.


In [15]:
import os
import math
import pandas as pd

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'

# 1. Load verified pairs and structure mapping metadata
df_pairs = pd.read_csv(os.path.join(processed_dir, 'verified_pairs.csv'))
df_struct = pd.read_csv(os.path.join(processed_dir, 'structure_mapping.csv'))

# Create lookup dictionary for structure metadata by CARD_model
struct_map = df_struct.set_index('CARD_model').to_dict('index')

# Function to parse C-alpha coordinates from a PDB file
def load_ca_coords(pdb_path):
    coords = {}
    if not pdb_path or not isinstance(pdb_path, str) or not os.path.exists(pdb_path):
        return coords
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                atom_name = line[12:16].strip()
                if atom_name == 'CA':
                    res_num_str = line[22:26].strip()
                    try:
                        res_num = int(res_num_str)
                        x = float(line[30:38])
                        y = float(line[38:46])
                        z = float(line[46:54])
                        if res_num not in coords:
                            coords[res_num] = (x, y, z)
                    except ValueError:
                        continue
    return coords

# Cache loaded PDB C-alpha coordinates per structure_id
ca_cache = {}

retained_3d_pairs = []
excluded_step6 = []

for idx, row in df_pairs.iterrows():
    pair_id = row['Pair_id']
    m_name = row['CARD_model']
    pos1 = int(row['Pos_1'])
    pos2 = int(row['Pos_2'])

    s_info = struct_map.get(m_name, {})
    struct_id = s_info.get('Structure_id')
    struct_source = s_info.get('Structure_source')
    pdb_path = s_info.get('PDB_path')
    chain = s_info.get('Chain', 'A')

    if not struct_id or not pdb_path or pd.isna(pdb_path):
        excluded_step6.append({
            'Pair_id': pair_id,
            'Reason_excluded': '3D structure unavailable for protein model',
            'CARD_model': m_name
        })
        continue

    if struct_id not in ca_cache:
        ca_cache[struct_id] = load_ca_coords(pdb_path)

    coords_dict = ca_cache[struct_id]

    c1 = coords_dict.get(pos1)
    c2 = coords_dict.get(pos2)

    if c1 is None or c2 is None:
        missing_res = []
        if c1 is None: missing_res.append(f"Pos1 ({pos1})")
        if c2 is None: missing_res.append(f"Pos2 ({pos2})")
        excluded_step6.append({
            'Pair_id': pair_id,
            'Reason_excluded': f"C-alpha coordinate missing in structure for {', '.join(missing_res)}",
            'CARD_model': m_name
        })
        continue

    # Calculate Sequence distance and 3D Euclidean C-alpha distance
    seq_dist = abs(pos1 - pos2)
    dist_3d = math.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2 + (c1[2] - c2[2])**2)

    row_dict = row.to_dict()
    row_dict['Structure_id'] = struct_id
    row_dict['Structure_source'] = struct_source
    row_dict['Chain'] = chain
    row_dict['Sequence_distance'] = seq_dist
    row_dict['3D_distance'] = round(dist_3d, 3)

    retained_3d_pairs.append(row_dict)

df_3d = pd.DataFrame(retained_3d_pairs)
df_ex6 = pd.DataFrame(excluded_step6)

df_3d.to_csv(os.path.join(processed_dir, 'struct_distance_pairs.csv'), index=False)
df_ex6.to_csv(os.path.join(processed_dir, 'excluded_pairs_step6.csv'), index=False)

print(f"Verified mutation pairs evaluated: {len(df_pairs)}")
print(f"Pairs successfully mapped in 3D structure: {len(df_3d)}")
print(f"Pairs excluded in Step 6: {len(df_ex6)}")
if not df_ex6.empty:
    print("\nExclusion breakdown in Step 6:")
    print(df_ex6['Reason_excluded'].value_counts().to_string())
if not df_3d.empty:
    print("\nSample calculated distance record:")
    print(df_3d.head(5)[['Pair_id', 'Protein/gene', 'WT_1', 'Pos_1', 'WT_2', 'Pos_2', 'Sequence_distance', '3D_distance', 'Structure_id']].to_string())

Verified mutation pairs evaluated: 113398
Pairs successfully mapped in 3D structure: 112882
Pairs excluded in Step 6: 516

Exclusion breakdown in Step 6:
Reason_excluded
3D structure unavailable for protein model    516

Sample calculated distance record:
     Pair_id                                                            Protein/gene WT_1  Pos_1 WT_2  Pos_2  Sequence_distance  3D_distance Structure_id
0  PAIR_0001                                                                    basR    L    157    L     71                 86       24.773    AF-Q9HV32
1  PAIR_0002     Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin    N    609    S    575                 34       16.859    AF-H8ZYN4
2  PAIR_0013  Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473    A    477                  4        6.253    AF-Q6GJC6
3  PAIR_0014  Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473    H    481                  8 

In [18]:
import os
import math
import random
import pandas as pd

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'

# 1. Load 3D distance records and structure mapping table
df_obs = pd.read_csv(os.path.join(processed_dir, 'struct_distance_pairs.csv'))
df_struct = pd.read_csv(os.path.join(processed_dir, 'structure_mapping.csv'))
struct_map = df_struct.set_index('CARD_model').to_dict('index')

# Function to parse C-alpha coordinates
def load_ca_coords(pdb_path):
    coords = {}
    if not pdb_path or not isinstance(pdb_path, str) or not os.path.exists(pdb_path):
        return coords
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                atom_name = line[12:16].strip()
                if atom_name == 'CA':
                    res_num_str = line[22:26].strip()
                    try:
                        res_num = int(res_num_str)
                        x = float(line[30:38])
                        y = float(line[38:46])
                        z = float(line[46:54])
                        if res_num not in coords:
                            coords[res_num] = (x, y, z)
                    except ValueError:
                        continue
    return coords

# 2. Pre-cache structure coordinates and pre-index sequence distances per structure
print("Caching structure coordinates and indexing residue pairs by sequence distance...")
ca_cache = {}
struct_seqdist_index = {}

for m_name, s_info in struct_map.items():
    s_id = s_info.get('Structure_id')
    p_path = s_info.get('PDB_path')

    # Check for non-null string path
    if pd.notna(s_id) and isinstance(p_path, str) and pd.notna(p_path) and os.path.exists(p_path):
        coords = load_ca_coords(p_path)
        if coords:
            ca_cache[s_id] = coords

            res_list = sorted(coords.keys())
            dist_dict = {}
            for i in range(len(res_list)):
                for j in range(i + 1, len(res_list)):
                    r1, r2 = res_list[i], res_list[j]
                    sd = abs(r1 - r2)
                    dist_dict.setdefault(sd, []).append((r1, r2))
            struct_seqdist_index[s_id] = dist_dict

# 3. Generate matched random control pairs (fixed seed = 42)
random.seed(42)
control_records = []

for idx, row in df_obs.iterrows():
    pair_id = row['Pair_id']
    s_id = row['Structure_id']
    seq_dist = int(row['Sequence_distance'])

    coords = ca_cache.get(s_id)
    dist_dict = struct_seqdist_index.get(s_id)

    if not coords or not dist_dict:
        continue

    # Select candidate pairs with exact sequence distance match
    candidates = dist_dict.get(seq_dist)

    if not candidates:
        # Fallback to nearest available sequence distance if exact is unavailable
        all_sds = sorted(dist_dict.keys())
        if not all_sds:
            continue
        nearest_sd = min(all_sds, key=lambda x: abs(x - seq_dist))
        candidates = dist_dict[nearest_sd]

    ctrl_p1, ctrl_p2 = random.choice(candidates)
    c1, c2 = coords[ctrl_p1], coords[ctrl_p2]
    ctrl_3d_dist = math.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2 + (c1[2] - c2[2])**2)
    ctrl_seq_dist = abs(ctrl_p1 - ctrl_p2)

    control_records.append({
        'Pair_id': pair_id,
        'Control_Pos_1': ctrl_p1,
        'Control_Pos_2': ctrl_p2,
        'Control_Sequence_distance': ctrl_seq_dist,
        'Control_3D_distance': round(ctrl_3d_dist, 3)
    })

df_ctrl = pd.DataFrame(control_records)
df_ctrl.to_csv(os.path.join(processed_dir, 'matched_controls.csv'), index=False)

print(f"Generated sequence-matched random controls for {len(df_ctrl)} observed pairs.")
print("\nSample matched control record:")
print(df_ctrl.head(5).to_string())

Caching structure coordinates and indexing residue pairs by sequence distance...
Generated sequence-matched random controls for 112882 observed pairs.

Sample matched control record:
     Pair_id  Control_Pos_1  Control_Pos_2  Control_Sequence_distance  Control_3D_distance
0  PAIR_0001             29            115                         86               18.959
1  PAIR_0002             26             60                         34               51.722
2  PAIR_0013            564            568                          4                6.249
3  PAIR_0014            502            510                          8               15.516
4  PAIR_0015            458            465                          7               13.283


In [19]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'

# 1. Load observed 3D distances and matched controls
df_obs = pd.read_csv(os.path.join(processed_dir, 'struct_distance_pairs.csv'))
df_ctrl = pd.read_csv(os.path.join(processed_dir, 'matched_controls.csv'))

# Merge on Pair_id
df_merged = pd.merge(df_obs, df_ctrl, on='Pair_id', how='inner')

obs_dist = df_merged['3D_distance'].values
ctrl_dist = df_merged['Control_3D_distance'].values
diff = obs_dist - ctrl_dist

# 2. Summary Statistics
obs_mean, obs_median = np.mean(obs_dist), np.median(obs_dist)
ctrl_mean, ctrl_median = np.mean(ctrl_dist), np.median(ctrl_dist)
diff_mean, diff_median, diff_std = np.mean(diff), np.median(diff), np.std(diff, ddof=1)

# 3. Statistical Testing
t_stat, p_val_ttest = stats.ttest_rel(obs_dist, ctrl_dist)
wilc_stat, p_val_wilc = stats.wilcoxon(obs_dist, ctrl_dist)

# Effect size (Cohen's d for paired samples)
cohen_d = diff_mean / diff_std if diff_std != 0 else 0.0

results = {
    'total_pairs': len(df_merged),
    'observed_3D_mean_A': round(float(obs_mean), 3),
    'observed_3D_median_A': round(float(obs_median), 3),
    'control_3D_mean_A': round(float(ctrl_mean), 3),
    'control_3D_median_A': round(float(ctrl_median), 3),
    'difference_mean_A': round(float(diff_mean), 3),
    'difference_median_A': round(float(diff_median), 3),
    'paired_ttest_statistic': round(float(t_stat), 4),
    'paired_ttest_pvalue': float(p_val_ttest),
    'wilcoxon_statistic': float(wilc_stat),
    'wilcoxon_pvalue': float(p_val_wilc),
    'cohens_d': round(float(cohen_d), 4)
}

# Save results JSON
with open(os.path.join(processed_dir, 'primary_statistical_results.json'), 'w') as f:
    json.dump(results, f, indent=4)

print("=== STEP 12: PRIMARY STATISTICAL COMPARISON RESULTS ===")
print(f"Total Paired Observations: {results['total_pairs']}")
print(f"Observed 3D Distance (Å): Mean = {results['observed_3D_mean_A']:.3f}, Median = {results['observed_3D_median_A']:.3f}")
print(f"Control 3D Distance (Å):  Mean = {results['control_3D_mean_A']:.3f}, Median = {results['control_3D_median_A']:.3f}")
print(f"Mean Difference (Obs - Ctrl): {results['difference_mean_A']:.3f} Å")
print(f"Median Difference:           {results['difference_median_A']:.3f} Å")
print(f"Paired t-test: t = {results['paired_ttest_statistic']}, p-value = {results['paired_ttest_pvalue']:.4e}")
print(f"Wilcoxon signed-rank test: p-value = {results['wilcoxon_pvalue']:.4e}")
print(f"Cohen's d effect size: {results['cohens_d']:.4f}")

=== STEP 12: PRIMARY STATISTICAL COMPARISON RESULTS ===
Total Paired Observations: 112882
Observed 3D Distance (Å): Mean = 20.313, Median = 18.455
Control 3D Distance (Å):  Mean = 22.922, Median = 20.856
Mean Difference (Obs - Ctrl): -2.609 Å
Median Difference:           -1.680 Å
Paired t-test: t = -72.4001, p-value = 0.0000e+00
Wilcoxon signed-rank test: p-value = 0.0000e+00
Cohen's d effect size: -0.2155


In [20]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

processed_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/processed'
results_dir = '/content/drive/MyDrive/AMR_Structural_Mutation_Project/results'
fig_dir = os.path.join(results_dir, 'figures')
os.makedirs(fig_dir, exist_ok=True)

# 1. Load merged dataset
df_obs = pd.read_csv(os.path.join(processed_dir, 'struct_distance_pairs.csv'))
df_ctrl = pd.read_csv(os.path.join(processed_dir, 'matched_controls.csv'))
df_merged = pd.merge(df_obs, df_ctrl, on='Pair_id')

sns.set_theme(style="whitegrid", font_scale=1.1)

# Plot 1: 3D Distance Distribution Density Plot
plt.figure(figsize=(8, 5))
sns.kdeplot(df_merged['3D_distance'], label='Observed Co-required Pairs', fill=True, alpha=0.35, color='#d95f02', linewidth=2)
sns.kdeplot(df_merged['Control_3D_distance'], label='Sequence-Matched Controls', fill=True, alpha=0.35, color='#7570b3', linewidth=2)
plt.axvline(df_merged['3D_distance'].median(), color='#d95f02', linestyle='--', label=f'Observed Median ({df_merged["3D_distance"].median():.2f} Å)')
plt.axvline(df_merged['Control_3D_distance'].median(), color='#7570b3', linestyle='--', label=f'Control Median ({df_merged["Control_3D_distance"].median():.2f} Å)')
plt.title('3D Spatial Distance Distribution of Co-required Resistance Pairs', fontsize=12, fontweight='bold')
plt.xlabel('3D C-alpha Distance (Å)')
plt.ylabel('Density')
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '3d_distance_distribution.png'), dpi=300)
plt.close()

# Plot 2: 3D Distance across Sequence Distance Bins
bins = [0, 10, 25, 50, 100, 200, 2000]
labels = ['1-10', '11-25', '26-50', '51-100', '101-200', '>200']
df_merged['Seq_Dist_Bin'] = pd.cut(df_merged['Sequence_distance'], bins=bins, labels=labels)

bin_stats = df_merged.groupby('Seq_Dist_Bin', observed=False)[['3D_distance', 'Control_3D_distance']].agg(['mean', 'count']).reset_index()

plt.figure(figsize=(9, 5))
x = np.arange(len(labels))
width = 0.35

means_obs = bin_stats[('3D_distance', 'mean')].values
means_ctrl = bin_stats[('Control_3D_distance', 'mean')].values

plt.bar(x - width/2, means_obs, width, label='Observed Co-required Pairs', color='#d95f02', alpha=0.85)
plt.bar(x + width/2, means_ctrl, width, label='Sequence-Matched Controls', color='#7570b3', alpha=0.85)

plt.xlabel('Sequence Separation Range (Residues)')
plt.ylabel('Mean 3D C-alpha Distance (Å)')
plt.title('Mean Spatial Distance Stratified by Sequence Separation Bins', fontsize=12, fontweight='bold')
plt.xticks(x, labels)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '3d_distance_by_seq_bin.png'), dpi=300)
plt.close()

# Export stratification table
bin_export = pd.DataFrame({
    'Seq_Bin': labels,
    'Pair_Count': bin_stats[('3D_distance', 'count')].values,
    'Observed_Mean_3D_A': np.round(means_obs, 3),
    'Control_Mean_3D_A': np.round(means_ctrl, 3),
    'Delta_Mean_A': np.round(means_obs - means_ctrl, 3)
})
bin_export.to_csv(os.path.join(results_dir, 'sequence_bin_stratification.csv'), index=False)

print("Figures successfully generated and saved to:", fig_dir)
print("\nSequence Separation Stratification Breakdown:")
print(bin_export.to_string(index=False))

Figures successfully generated and saved to: /content/drive/MyDrive/AMR_Structural_Mutation_Project/results/figures

Sequence Separation Stratification Breakdown:
Seq_Bin  Pair_Count  Observed_Mean_3D_A  Control_Mean_3D_A  Delta_Mean_A
   1-10       15045              10.038             10.767        -0.729
  11-25       15819              18.061             19.754        -1.692
  26-50       20261              17.939             20.079        -2.140
 51-100       29291              20.990             23.365        -2.375
101-200       22767              22.335             25.300        -2.965
   >200        9699              38.097             45.968        -7.871
